In [1]:
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END

In [2]:
class State(TypedDict):
    text: str
    feedback: str
    attempts: Annotated[int, operator.add]

In [3]:
# Node 1: Modify text
def write(state: State) -> str:
    new_text = state['text'] + "!"
    return {
        "text": new_text,
        "attempts": state['attempts'] + 1
    }

In [4]:
# Node 2: Evaluate text
def critic(state: State):
    if len(state['text']) >= 8:
        return {
            "feedback": "GOOD"    
        }
    return {"feedback": "RETRY"}

In [5]:
# Routing function
def should_continue(state: State) -> str:
    if state["feedback"] == "GOOD":
        return END

    return "write"

In [6]:
# Build graph
builder = StateGraph(State)

builder.add_node("write", write)
builder.add_node("critic", critic)

builder.add_edge(START, "write")
builder.add_edge("write", "critic")

builder.add_conditional_edges(
    "critic",
    should_continue,
    {
        "write": "write",
        END: END
    }
)

In [8]:
graph = builder.compile()

In [9]:
final = graph.invoke({
    "text": "Hello",
    "feedback": "",
    "attempts": 0
})

print(final["text"], final["feedback"], final["attempts"])

Hello!!! GOOD 7
